# Actividad (Parte 5) Explicabilidad usando contraejemplos
## Materia: Inteligencia Artificial Explicable - MSc. Inteligencia Artificial
Author: Esteban García-Cuesta, Departamento de Inteligencia Artificial, UPM (License CC-BY-NC)

This code has been developed to be used exclusively for educational purposes.

## Introducción
La explicabilidad con contraejemplos permite conocer el funcionamiento del modelo modificando las entradas e identificar las opciones posibles para revertir una decisión no favorable. En esta actividad utilizaras la base de datos Heart Disease UCI Machine Learning Repository.

## Objetivos:
  - Aprender como obtener un contraejemplo dado un caso
  - Aprender a obtener un mejor contraejemplo usando el parámetro $\lambda$ de la fórmula de Watcher $\lambda (f_{\theta}(x')-y')+d(x^e-x')$
  - Aprender a aplicar en un caso de reversión de decisión las técnicas de contraejemplo e interpretar los resultados

## Para hacer
  - Realiza los cambios en el código necesarios de acuerdo a las instrucciones de la actividad y responde a las preguntas que se indican en dichas instrucciones.

In [1]:
!pip install ucimlrepo mlxtend pandas scikit-learn numpy

In [2]:
#Lectura de los datos desde el repositorio UCI
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from mlxtend.evaluate import create_counterfactual
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [3]:
# fetch dataset
heart_disease = fetch_ucirepo(id=45)

# data (as pandas dataframes)
dfX = heart_disease.data.features
dfY = heart_disease.data.targets
df = pd.concat([dfX, dfY], axis=1)
df = df.dropna()
dfX = df[['age','trestbps','chol','thalach','oldpeak','ca']]
dfY = df['num']
dfX.head()


,age,trestbps,chol,thalach,oldpeak,ca
0,63,145,233,150,2.3,0.0
1,67,160,286,108,1.5,3.0
2,67,120,229,129,2.6,2.0
3,37,130,250,187,3.5,0.0
4,41,130,204,172,1.4,0.0


In [4]:
# Aprendizaje del modelo random forest
X = dfX.values
y = np.ravel(dfY.values)

clf = RandomForestClassifier(random_state=0)
clf.fit(X, y)

RandomForestClassifier(random_state=0)

## (a) Aumento de $\lambda$
Implementa un código (utilizando la librería mlxtend) que aumente el párametro $\lambda$ (según la formula vista en clase de Watcher) hasta que la diferencia entre la probabilidad que se predice para el contraejemplo de la clase deseada sea mayor de 0.50 (utiliza pasos de 1 para los distintos valores de $\lambda$ en el rango de 1 a 20)

La fórmula de Wachter es:

$$arg~min_{x^{\prime}}\lambda(f_{\theta}(x^{\prime})-y^{\prime})^{2}+d(x^{e},x^{\prime})$$

Esta ecuación busca el contrafactual $x'$ minimizando dos cosas:
- la diferencia con la predicción deseada (primer término)
- la distancia con el dato original (segundo término).

La función `create_counterfactual` ya contiene el algoritmo matemático para minimizar esta ecuación, así que nosotros solo le pasamos el valor de lammbda como argumento.

Por tanto, vamos a crear una función que automatice la búsqueda del mejor contraejemplo mediante una estrategia iterativa.

1. **Iteramos sobre $\lambda$**: ejecutamos un bucle variando el parámetro de regularización $\lambda$ desde 1 hasta 20. Este parámetro controla el equilibrio entre acercarse a la predicción deseada y mantener la similitud con el dato original

2. **Delegamos la optimización**: en cada paso del bucle, llamamos a `mlxtend` para resolver la fórmula de minimización de Wachter.

3. **Validamos el éxito**: verificamos si el contraejemplo generado logra que el modelo prediga la clase objetivo ("No Riesgo") con una probabilidad superior a 0.50

4. **Seleccionamos el mejor**: se detiene y devuelve el primer contraejemplo válido que encuentra (el que requiere el menor $\lambda$ posible para cruzar la frontera de decisión).

In [5]:
def buscar_contraejemplo_optimo(indice_instancia, modelo, X_dataset, y_dataset):

    x_ref = X_dataset[indice_instancia]
    clase_real = y_dataset[indice_instancia]

    # Nombres de las características del dataset
    nombres_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']

    print(f"\n--- Analizando Instancia {indice_instancia} (Clase Real: {clase_real}) ---")
    print(f"Características originales: {x_ref}")

    # Si ya es clase 0, no necesitamos contraejemplo para cambiar la decisión
    pred_actual = modelo.predict([x_ref])[0]
    if pred_actual == 0:
        print("La instancia ya está clasificada como 0 (sin riesgo).")
        return None

    # Bucle para encontrar el lambda óptimo
    for l in range(1, 21):
        cf = create_counterfactual(
            x_reference=x_ref,
            y_desired=0,
            model=modelo,
            X_dataset=X_dataset,
            y_desired_proba=1.0,
            lammbda=l,
            random_seed=123
        )

        # Evaluar la probabilidad que da el modelo al contraejemplo generado
        if cf is not None:
            probas = modelo.predict_proba(cf.reshape(1, -1))[0]
            prob_clase_0 = probas[0] # Probabilidad de la clase 0

            if prob_clase_0 > 0.50:
                print(f"Lambda necesario = {l}")
                print(f"Contraejemplo encontrado: {cf}")
                print(f"Probabilidad Clase 0: {prob_clase_0:.2%}")
                print(f"Cambios sugeridos para revertir la decisión: {cf - x_ref}")
                diferencia = cf - x_ref
                cambios_encontrados = False
                for i, feature in enumerate(nombres_features):
                    # Solo mostramos cambios significativos
                    if abs(diferencia[i]) > 0.001:
                        print(f"  - {feature}: {x_ref[i]:.2f} -> {cf[i]:.2f} (Delta: {diferencia[i]:+.2f})")
                        cambios_encontrados = True

                if not cambios_encontrados:
                    print("  (Cambios imperceptibles numéricamente)")

                return cf, l, prob_clase_0

    print("No se encontró contraejemplo válido en el rango de lambda 1-20.")
    return None, None, None

## (b) Aplicación a casos concretos
Examina los siguientes casos (2, 55, 65) y encuentra el contraejemplo que pase a no tener riesgo (clase 0) para cada uno de ellos siguiente el criterio del apartado (a) (es decir el más cercano que tenga una probabilidad de al menos 0.5 para la clase deseada). En base a los resultados obtenidos ¿Puedes extraer alguna conclusión sobre el modelo? ¿Qué pasa en las explicaciones cuando lambda es muy grande? Explica los resultados.

Ahora, vamos a iterar sobre los diferentes casos propuestos para encontrar el contraejemplo que pase a no tener riesgo y observar qué cambios sugiere el modelo para eliminar el riesgo cardíaco.

In [6]:
pacientes_ids = [2, 55, 65]
for idx in pacientes_ids:
    buscar_contraejemplo_optimo(idx, clf, X, y)


--- Analizando Instancia 2 (Clase Real: 1) ---
Características originales: [ 67.  120.  229.  129.    2.6   2. ]
Lambda necesario = 1
Contraejemplo encontrado: [ 6.70009837e+01  1.20000006e+02  2.28999990e+02  1.29000064e+02
  3.67825573e-04 -1.32308800e-04]
Probabilidad Clase 0: 62.00%
Cambios sugeridos para revertir la decisión: [ 9.83707451e-04  6.10746619e-06 -9.71756333e-06  6.40961471e-05
 -2.59963217e+00 -2.00013231e+00]
  - oldpeak: 2.60 -> 0.00 (Delta: -2.60)
  - ca: 2.00 -> -0.00 (Delta: -2.00)

--- Analizando Instancia 55 (Clase Real: 1) ---
Características originales: [ 54.  124.  266.  109.    2.2   1. ]
Lambda necesario = 1
Contraejemplo encontrado: [ 5.39803423e+01  1.23999991e+02  2.65999967e+02  1.09500107e+02
 -3.92200090e-04  9.64418746e-04]
Probabilidad Clase 0: 57.00%
Cambios sugeridos para revertir la decisión: [-1.96577310e-02 -8.72501002e-06 -3.31031958e-05  5.00106679e-01
 -2.20039220e+00 -9.99035581e-01]
  - age: 54.00 -> 53.98 (Delta: -0.02)
  - thalach: 109

### ¿Puedes extraer alguna conclusión sobre el modelo?
**Dependencia crítica de `oldpeak` y `ca`**: en los tres casos, para lograr que el modelo cambie su predicción, ha sido necesario reducir drásticamente dos variables específicas:

- **`oldpeak`**: en todos los casos se redujo a 0.0 (o valores negativos muy cercanos a cero).

- **`ca`**: igualmente, se redujo de valores de 1.0 o 2.0 a 0.0.

Esto significa que el modelo ha aprendido que la presencia de depresión del segmento ST inducida por el ejercicio y la obstrucción visible de vasos son los indicadores más fuertes de enfermedad cardíaca. Si estos síntomas desaparecen (valores a 0), el modelo tiende a predecir "Sin Riesgo" casi inmediatamente.

Observamos que variables como la edad (`age`) o la presión arterial (`trestbps`) apenas sufrieron cambios significativos para lograr el contraejemplo. Para este modelo, tener una edad avanzada (67, 54, 60 años) no condena al paciente a tener riesgo siempre y cuando sus indicadores clínicos (`oldpeak`, `ca`) estén sanos. El modelo pondera mucho más los síntomas activos que los factores de riesgo pasivos como la edad.

Por último, para los pacientes 2 y 55 (Clase Real 1 - Riesgo Bajo), bastó con un Lambda = 1 y modificar principalmente `oldpeak` y `ca`. Para el paciente 65 (Clase Real 2 - Riesgo Moderado), el algoritmo necesitó un Lambda = 2 (un "esfuerzo" ligeramente mayor) y, además de eliminar `oldpeak` y `ca`, tuvo que reducir significativamente el Colesterol (`chol`) en 41.5 puntos. Por tanto, cuanto mayor es la clase de riesgo original, más variables necesita "sanar" el contraejemplo para convencer al modelo, lo cual es coherente con la realidad médica.

### ¿Qué pasa en las explicaciones cuando lambda es muy grande? Explica los resultados

In [7]:
def experimento_lambda_alto(indice_instancia, modelo, X_dataset):
    x_ref = X_dataset[indice_instancia]
    print(f"--- EXPERIMENTO DE LAMBDA: Paciente {indice_instancia} ---\n")

    # Probamos dos valores extremos
    lambdas_a_probar = [1, 20] # 1 es el óptimo, 20 es el "muy alto"

    for l in lambdas_a_probar:
        # Generar contraejemplo
        cf = create_counterfactual(
            x_reference=x_ref,
            y_desired=0,
            model=modelo,
            X_dataset=X_dataset,
            y_desired_proba=1.0,
            lammbda=l,  # Aquí forzamos el valor
            random_seed=123
        )

        if cf is not None:
            # Obtener probabilidad y cambios
            prob = modelo.predict_proba(cf.reshape(1, -1))[0][0]
            diferencia = cf - x_ref
            distancia_L1 = sum(abs(diferencia)) # Suma total de cambios absolutos

            print(f"[Lambda = {l}]")
            print(f"  -> Probabilidad 'No Riesgo': {prob:.4f}")
            print(f"  -> Distancia Total (cuánto cambió el dato): {distancia_L1:.2f}")

            # Ver cambios específicos
            nombres = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
            print("  -> Cambios notables:")
            for i, feat in enumerate(nombres):
                if abs(diferencia[i]) > 0.001:
                    print(f"     * {feat}: {diferencia[i]:+.2f}")
            print("-" * 40)

# Ejecutamos el experimento con el Paciente 2
experimento_lambda_alto(2, clf, X)

--- EXPERIMENTO DE LAMBDA: Paciente 2 ---

[Lambda = 1]
  -> Probabilidad 'No Riesgo': 0.6200
  -> Distancia Total (cuánto cambió el dato): 4.60
  -> Cambios notables:
     * oldpeak: -2.60
     * ca: -2.00
----------------------------------------
[Lambda = 20]
  -> Probabilidad 'No Riesgo': 0.9100
  -> Distancia Total (cuánto cambió el dato): 34.10
  -> Cambios notables:
     * age: -10.50
     * chol: -0.50
     * thalach: +18.50
     * oldpeak: -2.60
     * ca: -2.00
----------------------------------------


Basado en el experimento realizado con el paciente 2, podemos observar claramente el impacto del parámetro de regularización $\lambda$ en el método de Wachter. Esta fórmula busca minimizar una función de pérdida que equilibra el error de predicción y la distancia al ejemplo original, donde $\lambda$ pondera la importancia del primer término.

Con un lambda bajo ($\lambda=1$), el algoritmo funcionó de manera eficiente, buscando el camino más corto para cruzar la frontera de decisión. Logró mantener una distancia total baja (4.60) modificando únicamente las variables críticas (`oldpeak` y `ca`), respetando así el principio de que los contraejemplos deben ser lo más similares posible al dato original.

Por el contrario, al aumentar el parámetro a un valor alto ($\lambda=20$), el método priorizó casi exclusivamente maximizar la probabilidad de la clase deseada (subiéndola del 62% al 91%), restando importancia a la cercanía con el ejemplo original. Esta amplificación del peso del error de predicción en la fórmula de Wachter provocó consecuencias negativas evidentes:
- **Aumento de la distancia**: la distancia total se disparó de 4.60 a 34.10, generando un contraejemplo muy alejado de la realidad del paciente.
- **Violación de la Sparsity**: el modelo comenzó a mover variables innecesarias para asegurar el resultado, modificando 5 variables en lugar de 2. Se añadieron cambios superfluos, como un aumento de 18.5 pulsaciones en el ritmo cardíaco (thalach) y modificaciones en el colesterol.
- **Pérdida de realismo**: el sistema sugirió cambios imposibles o absurdos, como reducir la edad del paciente en 10.5 años, simplemente para aumentar la probabilidad de clasificación.

En conclusión, un $\lambda$ demasiado grande puede generar explicaciones que son "perfectas" desde el punto de vista matemático (alta probabilidad), pero inútiles para el usuario, ya que violan los principios de generar cambios mínimos y accionables.